In [35]:
# Import thư viện
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, average_precision_score, brier_score_loss, confusion_matrix, f1_score, log_loss, precision_score, recall_score, roc_auc_score

In [36]:
# Đọc dữ liệu train, valid, test
project_root = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
processed_dir = project_root / "data" / "processed"
results_dir = project_root / "data" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(processed_dir / "train.csv")
valid_df = pd.read_csv(processed_dir / "valid.csv")
test_df = pd.read_csv(processed_dir / "test.csv")
train_valid_df = pd.concat([train_df, valid_df], ignore_index=True)

train_df.head()

,datetime,precipitation,relative_humidity_2m,surface_pressure,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,hour,month,...,cloud_cover_lag_2h,cloud_cover_lag_3h,temperature_2m_lag_1h,temperature_2m_lag_2h,temperature_2m_lag_3h,precipitation_rolling_3h,precipitation_rolling_6h,rain_next_1h,rain_next_2h,rain_next_3h
0,2019-01-01 05:00:00,0.0,64,1026.0,8.8,15.0,17,100,5,1,...,34.0,39.0,9.0,9.1,9.7,0.0,0.0,0,0,0
1,2019-01-01 06:00:00,0.0,64,1026.4,8.9,14.6,16,100,6,1,...,59.0,34.0,8.8,9.0,9.1,0.0,0.0,0,0,0
2,2019-01-01 07:00:00,0.0,71,1026.5,9.2,15.8,24,100,7,1,...,100.0,59.0,8.9,8.8,9.0,0.0,0.0,0,0,0
3,2019-01-01 08:00:00,0.0,67,1027.4,9.8,15.6,23,100,8,1,...,100.0,100.0,9.2,8.9,8.8,0.0,0.0,0,0,0
4,2019-01-01 09:00:00,0.0,65,1028.3,10.3,14.8,18,100,9,1,...,100.0,100.0,9.8,9.2,8.9,0.0,0.0,0,0,0


In [37]:
# Cấu hình target và threshold
targets = ["rain_next_1h", "rain_next_2h", "rain_next_3h"]
threshold = 0.5

targets

['rain_next_1h', 'rain_next_2h', 'rain_next_3h']

In [38]:
# Hàm dự đoán xác suất theo month và hour
def predict_by_month_hour(train_data, data, target):
    table = train_data.groupby(["month", "hour"])[target].mean().reset_index(name="prob")
    default_prob = train_data[target].mean()

    pred = data[["datetime", "month", "hour", target]].merge(table, on=["month", "hour"], how="left")
    pred["prob"] = pred["prob"].fillna(default_prob)
    pred["pred"] = (pred["prob"] >= threshold).astype(int)
    return pred

In [39]:
# Hàm tính chỉ số đánh giá
def evaluate(y_true, y_prob, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "log_loss": log_loss(y_true, y_prob, labels=[0, 1]),
        "brier_score": brier_score_loss(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }

In [40]:
# Chạy baseline và lưu kết quả
rows = []
test_predictions = test_df[["datetime", "month", "hour"]].copy()

for target in targets:
    pred_df = predict_by_month_hour(train_valid_df, test_df, target)
    metric = evaluate(pred_df[target], pred_df["prob"], pred_df["pred"])

    metric["baseline"] = "month_hour"
    metric["split"] = "test"
    metric["target"] = target
    metric["threshold"] = threshold
    rows.append(metric)

    test_predictions[f"{target}_true"] = pred_df[target].values
    test_predictions[f"{target}_prob"] = pred_df["prob"].values
    test_predictions[f"{target}_pred"] = pred_df["pred"].values

metrics_df = pd.DataFrame(rows)
metrics_df = metrics_df[["baseline", "split", "target", "threshold", "log_loss", "brier_score", "roc_auc", "pr_auc", "accuracy", "precision", "recall", "f1", "tn", "fp", "fn", "tp"]]

metrics_df.to_csv(results_dir / "baseline_month_hour_metrics.csv", index=False)
test_predictions.to_csv(results_dir / "baseline_month_hour_test_predictions.csv", index=False)

old_valid_file = results_dir / "baseline_month_hour_valid_predictions.csv"
if old_valid_file.exists():
    old_valid_file.unlink()

metrics_df

,baseline,split,target,threshold,log_loss,brier_score,roc_auc,pr_auc,accuracy,precision,recall,f1,tn,fp,fn,tp
0,month_hour,test,rain_next_1h,0.5,0.540490,0.179886,0.677139,0.430859,0.74489,0.589837,0.139306,0.225381,6198,226,2008,325
1,month_hour,test,rain_next_2h,0.5,0.540383,0.179855,0.677319,0.431016,0.74489,0.589837,0.139306,0.225381,6198,226,2008,325
2,month_hour,test,rain_next_3h,0.5,0.540238,0.179814,0.677603,0.431153,0.74489,0.589837,0.139306,0.225381,6198,226,2008,325
